# NIH ChestX-ray14 Multi-label Classification

## Project Goal

The goal of this project is to build a deep learning workflow for multi-label thoracic disease classification using the NIH ChestX-ray14 dataset and PyTorch.

The project will begin with dataset inspection and exploratory analysis before moving into patient-level data splitting, image preprocessing, PyTorch datasets and dataloaders, transfer learning, model evaluation, threshold selection, error analysis, explainability, and assessment of clinical usefulness.

## Data Source

This project uses the **NIH ChestX-ray14** dataset, a large publicly available chest X-ray dataset released by the National Institutes of Health Clinical Center.

The dataset contains:

- **112,120 frontal chest X-ray images**
- **30,805 patients**
- labels for **14 thoracic disease categories**
- patient and image metadata
- disease labels derived from associated radiology reports

The dataset will be used for multi-label classification, meaning that a single chest X-ray may contain more than one labeled finding.

### Dataset Citation

Wang, X., Peng, Y., Lu, L., Lu, Z., Bagheri, M., & Summers, R. M. (2017).  
*ChestX-ray8: Hospital-scale chest X-ray database and benchmarks on weakly-supervised classification and localization of common thorax diseases.*  
Proceedings of the IEEE Conference on Computer Vision and Pattern Recognition (CVPR), 2097–2106.

The NIH ChestX-ray14 dataset is an expanded release of the dataset described in this publication.

---------------------------------------------------

## Initial Dataset Inspection

Begin by reviewing the dataset dimensions, column structure, data types, and missingness before any cleaning or feature engineering.

In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np

import json

In [ ]:
# Use a relative path so the notebook remains portable within the repository.
data_dir = Path("../data")

df = pd.read_csv(data_dir / "Data_Entry_2017_v2020.csv")

df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe(include="all")

In [ ]:
df.columns.tolist()

In [ ]:
df = df.rename(
    columns={
        "Image Index": "image_index",
        "Finding Labels": "finding_labels",
        "Follow-up #": "follow_up",
        "Patient ID": "patient_id",
        "Patient Age": "patient_age",
        "Patient Sex": "patient_sex",
        "View Position": "view_position",
        "OriginalImage[Width": "image_width",
        "Height]": "image_height",
        "OriginalImagePixelSpacing[x": "pixel_spacing_x",
        "y]": "pixel_spacing_y",
    }
)

df.head()

In [ ]:
df.duplicated().sum()

In [ ]:
df["image_index"].duplicated().sum()

In [ ]:
df.nunique().sort_values(ascending=False)

In [ ]:
df["patient_sex"].value_counts()

In [ ]:
df["view_position"].value_counts()

## Patient-Level Structure

Because individual patients may have multiple chest X-rays, the distribution of images per patient is examined before label analysis and model splitting. 

Patient-level separation will be necessary later to prevent images from the same patient appearing in both training and evaluation sets.

In [ ]:
images_per_patient = df.groupby("patient_id").size()

images_per_patient.describe()

In [ ]:
images_per_patient.value_counts().sort_index().head(20)

In [ ]:
import matplotlib.pyplot as plt

images_per_patient.hist(bins=50)

plt.xlabel("Images per patient")
plt.ylabel("Number of patients")
plt.title("Distribution of Images per Patient")
plt.show()

Most patients have relatively few images, with a median of one image per patient and 75% of patients having three or fewer images. However, a small number of patients have many repeated examinations, with as many as 184 images for a single patient.

Because multiple images can belong to the same patient, train, validation, and test sets must be separated at the patient level to prevent information leakage between model development and evaluation.

## Finding Label Structure

The dataset uses multi-label annotations, meaning a single chest X-ray may contain more than one finding. The label structure and prevalence of individual findings are examined before modeling to understand class imbalance and disease co-occurrence.

In [ ]:
df["finding_labels"].value_counts().head(10)

In [ ]:
# Extract every unique finding represented in the pipe-delimited label strings
all_labels = sorted(
    {label for labels in df["finding_labels"].str.split("|") for label in labels}
)

all_labels

In [ ]:
for label in all_labels:
    df[label] = df["finding_labels"].str.contains(label, regex=False).astype(int)

In [ ]:
label_summary = pd.DataFrame(
    {"count": df[all_labels].sum(), "percent": df[all_labels].mean() * 100}
).sort_values("count", ascending=False)

label_summary = label_summary.round({"percent": 2})

label_summary

In [ ]:
df["num_findings"] = df[all_labels].sum(axis=1)
df["num_findings"].value_counts().sort_index()

The label distribution is highly imbalanced. More than half of the images are labeled `No Finding`, while several individual abnormalities occur in fewer than 3% of images. The rarest finding, `Hernia`, appears in only 0.2% of the dataset.

Most images contain a single label, although multi-label cases are common enough to require a true multi-label modeling approach rather than treating the problem as mutually exclusive classification.

This imbalance will need to be considered when selecting evaluation metrics and training strategies.

In [ ]:
# Exclude "No Finding" and count how often each pair of disease labels appears together.
disease_labels = [label for label in all_labels if label != "No Finding"]

cooccurrence = df[disease_labels].T.dot(df[disease_labels])

cooccurrence

In [ ]:
# Normalize pairwise overlap so common findings do not dominate the comparison.
label_counts = df[disease_labels].sum().to_numpy()
cooccurrence_array = cooccurrence.to_numpy()

union = label_counts[:, None] + label_counts[None, :] - cooccurrence_array

jaccard = cooccurrence_array / union

# The matrix is symmetric, so display each disease pair only once.
mask = np.triu(np.ones_like(jaccard, dtype=bool), k=0)
jaccard_masked = np.ma.masked_where(mask, jaccard)

fig, ax = plt.subplots(figsize=(7, 5))

im = ax.imshow(jaccard_masked, cmap="Blues")

ax.set_xticks(range(len(disease_labels)))
ax.set_yticks(range(len(disease_labels)))
ax.set_xticklabels(disease_labels, rotation=90)
ax.set_yticklabels(disease_labels)

fig.colorbar(im, ax=ax, label="Jaccard similarity")
ax.set_title("Normalized Disease Label Co-occurrence")

plt.tight_layout()
plt.show()

Several findings show meaningful overlap rather than occurring independently. The strongest normalized co-occurrence was observed between Effusion and Atelectasis, followed by Infiltration with Effusion and Atelectasis. Other notable relationships included Pneumothorax with Emphysema and Nodule with Mass.

These patterns reinforce the multi-label nature of the dataset, since clinically related findings may appear together on the same image.

## Image-Level EDA

The metadata describes the patients, acquisition characteristics, and disease labels, but the model will ultimately learn from the chest X-ray images themselves. Before defining preprocessing or PyTorch transforms, the image files should be inspected directly to understand their dimensions, intensity range, view differences, and overall visual quality.

In [ ]:
from PIL import Image

image_dir = data_dir / "images"

image_files = list(image_dir.glob("*.png"))

len(image_files)

In [ ]:
available_images = {path.name for path in image_files}

image_df = df[df["image_index"].isin(available_images)].copy()

image_df.shape

In [ ]:
sample_path = image_dir / image_df.iloc[0]["image_index"]

image = Image.open(sample_path)
image_array = np.array(image)

print("Filename:", sample_path.name)
print("Shape", image_array.shape)
print("Data type:", image_array.dtype)
print("Min pixel value:", image_array.min())
print("Max pixel value:", image_array.max())

In [ ]:
plt.figure(figsize=(6, 6))
plt.imshow(image_array, cmap="grey")
plt.title(image_df.iloc[0]["finding_labels"])
plt.axis("off")
plt.show()

In [ ]:
# Compare a small sample of normal and abnormal images before defining preprocessing.
sample_df = pd.concat(
    [
        image_df[image_df["finding_labels"] == "No Finding"].sample(4, random_state=42),
        image_df[image_df["finding_labels"] != "No Finding"].sample(4, random_state=42),
    ]
)

fig, axes = plt.subplots(2, 4, figsize=(14, 7))

for ax, (_, row) in zip(axes.flat, sample_df.iterrows()):
    image = Image.open(image_dir / row["image_index"])
    ax.imshow(image, cmap="gray")
    ax.set_title(row["finding_labels"], fontsize=9)
    ax.axis("off")

plt.tight_layout()
plt.show()

The sampled radiographs show meaningful variation in acquisition and image appearance, including PA and AP projections, differences in positioning and inspiration, variable contrast/exposure, and the presence of portable imaging artifacts such as leads and support devices.

These differences are clinically expected but may also provide non-pathologic cues to a deep learning model. Preprocessing and evaluation should therefore account for acquisition variability rather than assuming all images are visually standardized.

In [ ]:
# Sample image dimensions and intensity statistics to confirm image consistency
# and quantify variation in pixel values before defining preprocessing.

image_stats = []

for path in image_files[:500]:
    image = Image.open(path)
    image_array = np.array(image)

    image_stats.append(
        {
            "image_index": path.name,
            "height": image_array.shape[0],
            "width": image_array.shape[1],
            "min_pixel": image_array.min(),
            "max_pixel": image_array.max(),
            "mean_pixel": image_array.mean(),
        }
    )

image_stats_df = pd.DataFrame(image_stats)

image_stats_df.describe()

The sampled chest X-rays were consistently **1024 × 1024 pixels**, indicating that image dimensions are standardized within the downloaded batch.

Pixel intensities, however, varied substantially across images. Mean intensity ranged from approximately **93 to 195**, while maximum pixel values ranged from **160 to 255**. This confirms the visual differences in exposure and contrast observed in the sample images.

The images are therefore spatially consistent but not intensity-standardized. Image resizing and intensity normalization will be important preprocessing steps before model training.

### AP vs PA Projection

AP and PA chest radiographs differ in acquisition geometry, patient positioning, and often clinical context. A small sample of each projection is compared to understand whether projection-related variation should be considered during model development.

In [ ]:
ap_sample = image_df[image_df["view_position"] == "AP"].sample(3, random_state=42)
pa_sample = image_df[image_df["view_position"] == "PA"].sample(3, random_state=42)

projection_sample = pd.concat([ap_sample, pa_sample])

fig, axes = plt.subplots(2, 3, figsize=(6, 4))

for ax, (_, row) in zip(axes.flat, projection_sample.iterrows()):
    image = Image.open(image_dir / row["image_index"])

    ax.imshow(image, cmap="gray")
    ax.set_title(f"{row['view_position']} | {row['finding_labels']}", fontsize=9)
    ax.axis("off")

plt.tight_layout()
plt.show()

AP images show more variability in positioning and image quality than PA images, including poorer inspiration, rotation, and contrast differences in some cases.

This is clinically expected. AP chest radiographs are often obtained when a patient's condition does not allow a standard upright PA examination, such as with portable imaging or limited mobility. As a result, AP images may contain both technical differences and indirect signals of patient acuity.

This matters for modeling because the network could learn projection- or care-setting-related cues in addition to actual pathology. Projection type should therefore be considered during model development and evaluation.

## Exploratory Data Analysis Summary

The NIH ChestX-ray14 metadata contains **112,120 images from 30,805 patients**, with no missing values or duplicate image identifiers in the metadata reviewed. Although the median patient contributes only one image, some patients have more than 100 repeated examinations. Because multiple images may come from the same patient, train, validation, and test sets must be separated at the **patient level** to prevent information leakage.

The dataset is highly imbalanced. More than half of the images are labeled `No Finding`, while several abnormalities occur in fewer than 3% of images. Most images contain a single label, but a substantial number contain multiple abnormalities.

This makes the task a **multi-label classification problem** rather than a standard multiclass problem. The 14 disease labels are not mutually exclusive: one image can be positive for several findings at the same time. The model will therefore need to produce an independent probability for each disease rather than selecting a single class. This will affect the loss function, thresholding strategy, and evaluation metrics, and performance should be assessed by disease label rather than relying on a single overall accuracy value.

Disease labels also show meaningful co-occurrence, with several findings appearing together more frequently than others. This further supports treating the labels as related but non-exclusive outputs.

An important limitation is that the NIH disease labels are **weak labels derived automatically from associated radiology-report text rather than manually verified image-level annotations for every radiograph**. As a result, some labels may be missing, incorrect, or imprecise. Model performance will therefore partly reflect the quality of the label-extraction process rather than only the model's ability to recognize true radiographic findings. This limitation will need to be considered when interpreting both errors and apparent model performance.

Image-level inspection showed that the sampled chest radiographs are consistently **1024 × 1024 grayscale PNGs**, but their intensity distributions and visual appearance vary considerably. Differences were observed in contrast, positioning, inspiration, and acquisition technique.

AP images showed particularly noticeable variability compared with PA images. This is clinically expected because AP radiographs are often used when a patient's condition or mobility prevents a standard upright PA examination. As a result, projection type may contain information about both acquisition technique and patient condition, creating a risk that the model could learn non-pathologic cues in addition to disease features.

Overall, model development will need to account for **patient-level leakage, severe class imbalance, multi-label outputs, weak-label uncertainty, acquisition variability, and image intensity normalization**.

-----------------------------------------------

## Train, Validation, and Test Split

Because some patients contribute multiple radiographs, the dataset must be split at the **patient level** rather than the image level. All images from a given patient will remain within the same dataset partition. This prevents patient-level information leakage between training, validation, and test data and provides a more realistic estimate of model generalization to previously unseen patients.

The split also needs to account for the fact that this is a **multi-label and highly imbalanced dataset**. A purely random patient split could leave some rare disease labels poorly represented in the validation or test sets.

To support the split, the image-level disease indicators are therefore temporarily collapsed to **one row per patient**. For each disease, a value of `1` means that the patient had **at least one image** with that label, while `0` means that label never appeared for that patient.

This patient-level label summary will be used only to guide the train/validation/test split so that disease representation remains reasonably similar across partitions. The model itself will still be trained on individual radiographs using the original image-level labels.

In [ ]:
# Summarize whether each patient ever had each disease label.
patient_labels = df.groupby("patient_id")[disease_labels].max().reset_index()

patient_labels.head()

In [ ]:
# Check how many patients contribute to at least one positive sample of each disease
patient_label_summary = pd.DataFrame(
    {
        "patients": patient_labels[disease_labels].sum(),
        "percent": patient_labels[disease_labels].mean() * 100,
    }
).sort_values("patients", ascending=False)

patient_label_summary.round({"percent": 2})

The patient-level distribution remains highly imbalanced. Common findings such as Infiltration, Atelectasis, and Effusion are represented across thousands of patients, while rare findings such as Hernia occur in relatively few patients.

This matters for dataset splitting because a simple random patient split could produce validation or test sets with noticeably different disease prevalence, particularly for rare labels. The split should therefore preserve patient independence while also keeping the multi-label disease distribution as similar as reasonably possible across train, validation, and test sets.

For a multi-label problem, ordinary `train_test_split(..., stratify=...)` isn’t enough, because stratify expects essentially one class value per row. Here, each patient can have several disease labels simultaneously.

### Multi-Label Stratification

A standard stratified split assumes that each observation belongs to a single class. That does not fit this dataset because each patient may have several disease labels simultaneously.

To preserve the prevalence of multiple disease labels as closely as possible across the dataset partitions, **iterative multi-label stratification** is used. The algorithm distributes patients while considering the full set of disease indicators rather than stratifying on only one target.

The split is performed on the patient-level table, so each patient is assigned to exactly one partition. The original image-level dataset can then inherit the split assignment from its patient.

In [ ]:
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

# The splitter needs two things:
# - X: the observations being assigned to partitions — here, the patients
# - y: the multi-label disease pattern for each patient — the 14 binary disease columns

### Prepare Patient Data for Splitting

The split is performed using the patient-level table created above.

`X` contains the patient identifiers that will be assigned to train, validation, or test. `y` contains the 14 binary disease indicators for those same patients.

The disease indicators are supplied to the stratification algorithm so that it can distribute patients while keeping disease prevalence as similar as possible across the partitions.

In [ ]:
# values should match what is give in the methodology / data page (30805 & 14)
X = patient_labels["patient_id"].to_numpy()
y = patient_labels[disease_labels].to_numpy()

X.shape, y.shape

In [ ]:
# First split: 70% of patients for training, 30% held aside for validation and testing.
splitter = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.30, random_state=42)

train_idx, temp_idx = next(splitter.split(X, y))

train_patients = patient_labels.iloc[train_idx].copy()
temp_patients = patient_labels.iloc[temp_idx].copy()

len(train_patients), len(temp_patients)

### Split the Remaining Patients into Validation and Test Sets

The first stratified split reserved 70% of patients for training and held the remaining 30% aside.

The remaining patients are now split equally into validation and test sets. Because this second split is also multi-label stratified, it attempts to preserve the disease-label distribution within both final partitions.

This produces the intended overall allocation of approximately **70% training, 15% validation, and 15% test patients**.

In [ ]:
# Second split: divide the remaining 30% equally into validation and test sets
temp_X = temp_patients["patient_id"].to_numpy()
temp_y = temp_patients[disease_labels].to_numpy()

splitter_temp = MultilabelStratifiedShuffleSplit(
    n_splits=1, test_size=0.5, random_state=42
)

val_idx, test_idx = next(splitter_temp.split(temp_X, temp_y))

val_patients = temp_patients.iloc[val_idx].copy()
test_patients = temp_patients.iloc[test_idx].copy()

len(train_patients), len(val_patients), len(test_patients)

In [ ]:
# Confirm that patients are completely separated across train, validation, and test sets
train_ids = set(train_patients["patient_id"])
val_ids = set(val_patients["patient_id"])
test_ids = set(test_patients["patient_id"])

print("Train/Val overlap", len(train_ids & val_ids))
print("Train/Test overlap", len(train_ids & test_ids))
print("Test/Val overlap", len(test_ids & val_ids))

In [ ]:
# Compare patient-level disease prevelance across the final partition
split_prevalance = pd.DataFrame(
    {
        "train": train_patients[disease_labels].mean() * 100,
        "validation": val_patients[disease_labels].mean() * 100,
        "test": test_patients[disease_labels].mean() * 100,
    }
)

split_prevalance.round(2)

The final train, validation, and test partitions contain no overlapping patients, confirming that patient-level leakage has been prevented.

Disease prevalence is also nearly identical across all three partitions, including rare findings such as Hernia. This indicates that iterative multi-label stratification successfully preserved the patient-level label distribution while maintaining complete patient separation.

### Map Patient Splits Back to the Image-Level Dataset

The split was created at the patient level to prevent leakage, but model training will occur at the image level.

Each image is therefore assigned to the same partition as its patient. This preserves the patient-level separation while restoring the original image-level structure required for the PyTorch pipeline.

In [ ]:
# Assign each image to the same partition as its patient
df["split"] = "unassigned"

df.loc[df["patient_id"].isin(train_ids), "split"] = "train"
df.loc[df["patient_id"].isin(val_ids), "split"] = "validation"
df.loc[df["patient_id"].isin(test_ids), "split"] = "test"

df["split"].value_counts()

After mapping the patient assignments back to the original image-level dataset, the final image counts remain close to the intended **70/15/15** allocation.

This was not guaranteed because patients contribute different numbers of radiographs, but the resulting image distribution is still well balanced across train, validation, and test sets.

## Next Steps

The exploratory analysis and patient-level split are now complete.

The final image-level metadata, disease-label columns, and train/validation/test assignments are saved so the next notebook can begin directly with the PyTorch modeling pipeline without repeating the EDA or split logic.

The next notebook, **`02_baseline_model.ipynb`**, will use these saved outputs to:

- rebuild the training, validation, and test datasets
- define image preprocessing and DataLoaders
- configure the DenseNet-121 baseline model
- train and validate the initial classifier

In [ ]:
processed_dir = data_dir / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

# Save the image-level metadata, including disease columns and split assignments
df.to_csv(processed_dir / "metadata_with_splits.csv", index=False)

# Preserve the exact disease order used for targets and model outputs
with open(processed_dir / "disease_labels.json", "w") as file:
    json.dump(disease_labels, file, indent=2)